# Notebook 3: Breaking NYC into Regions (Improved)

This notebook upgrades region clustering with stronger quality checks and business-ready outputs.

## Goals
- Cluster NYC pickup locations into operational regions.
- Select `k` using quality metrics (elbow + silhouette + stability metrics).
- Assign each trip a `region_id` and `region_label`.
- Save artifacts needed for upcoming notebooks:
  - cluster summary and centroids
  - nearby-region distance map (for relocation logic)
  - region-hour profile and risk baseline
  - labeled trip data


In [ ]:
from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
# Project paths
project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

data_interim = project_root / "data" / "interim"
data_processed = project_root / "data" / "processed"
models_dir = project_root / "models"

for folder in [data_interim, data_processed, models_dir]:
    folder.mkdir(parents=True, exist_ok=True)

candidate_inputs = [
    data_interim / "location_data.csv",
    data_interim / "cleaned_data.csv",
    data_interim / "cleaned_data.parquet",
    data_processed / "cleaned_data.csv",
    data_processed / "cleaned_data.parquet",
]

input_path = next((path for path in candidate_inputs if path.exists()), None)

if input_path is None:
    raise FileNotFoundError(
        "No cleaned input file found. Expected one of: "
        + ", ".join(str(path) for path in candidate_inputs)
    )

print(f"Using input file: {input_path}")


In [ ]:
NYC_BOUNDS = {
    "min_lat": 40.55,
    "max_lat": 40.95,
    "min_lon": -74.10,
    "max_lon": -73.65,
}


def coordinate_chunk_reader(path: Path, chunksize: int = 250_000):
    cols = ["pickup_latitude", "pickup_longitude"]

    if path.suffix.lower() == ".csv":
        for chunk in pd.read_csv(path, usecols=lambda c: c in cols, chunksize=chunksize):
            yield chunk
    else:
        parquet_df = pd.read_parquet(path, columns=cols)
        for start in range(0, len(parquet_df), chunksize):
            yield parquet_df.iloc[start:start + chunksize].copy()


def full_chunk_reader(path: Path, columns=None, chunksize: int = 200_000):
    if path.suffix.lower() == ".csv":
        kwargs = {"chunksize": chunksize}
        if columns is not None:
            kwargs["usecols"] = lambda c: c in columns
        for chunk in pd.read_csv(path, **kwargs):
            yield chunk
    else:
        if columns is not None:
            parquet_df = pd.read_parquet(path, columns=columns)
        else:
            parquet_df = pd.read_parquet(path)
        for start in range(0, len(parquet_df), chunksize):
            yield parquet_df.iloc[start:start + chunksize].copy()


def clean_coordinates(frame: pd.DataFrame):
    coords = frame[["pickup_latitude", "pickup_longitude"]].copy()
    coords["pickup_latitude"] = pd.to_numeric(coords["pickup_latitude"], errors="coerce")
    coords["pickup_longitude"] = pd.to_numeric(coords["pickup_longitude"], errors="coerce")

    valid_mask = (
        coords["pickup_latitude"].between(NYC_BOUNDS["min_lat"], NYC_BOUNDS["max_lat"])
        & coords["pickup_longitude"].between(NYC_BOUNDS["min_lon"], NYC_BOUNDS["max_lon"])
    )

    return coords.loc[valid_mask].reset_index(drop=True), valid_mask


In [ ]:
# Build a representative sample without loading all rows at once
sample_parts = []
total_valid_points = 0

for chunk in coordinate_chunk_reader(input_path, chunksize=250_000):
    cleaned, _ = clean_coordinates(chunk)
    if cleaned.empty:
        continue

    total_valid_points += len(cleaned)
    take = min(4_000, len(cleaned))
    sample_parts.append(cleaned.sample(n=take, random_state=RANDOM_STATE))

if not sample_parts:
    raise ValueError("No valid pickup coordinate rows found after cleaning.")

sample_coords = pd.concat(sample_parts, ignore_index=True)

if len(sample_coords) > 200_000:
    sample_coords = sample_coords.sample(n=200_000, random_state=RANDOM_STATE).reset_index(drop=True)

print(f"Total valid coordinate rows found: {total_valid_points:,}")
print(f"Sample rows used for k-search and visual checks: {len(sample_coords):,}")
sample_coords.head()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
ax.set_facecolor("#0f172a")
ax.scatter(
    sample_coords["pickup_longitude"],
    sample_coords["pickup_latitude"],
    s=1,
    alpha=0.35,
    color="#22d3ee",
)
ax.set_title("Pickup Distribution (Sample)", fontsize=13)
ax.set_xlabel("pickup_longitude")
ax.set_ylabel("pickup_latitude")
plt.show()


In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    # Distance in kilometers between two geo points.
    radius_km = 6371.0088

    lat1, lon1, lat2, lon2 = map(
        np.radians,
        [lat1, lon1, lat2, lon2],
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    )
    c = 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    return radius_km * c



def pairwise_haversine_matrix(centroids_lat_lon: np.ndarray) -> np.ndarray:
    lat = np.radians(centroids_lat_lon[:, 0])
    lon = np.radians(centroids_lat_lon[:, 1])

    lat1 = lat[:, None]
    lat2 = lat[None, :]
    lon1 = lon[:, None]
    lon2 = lon[None, :]

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

    radius_km = 6371.0088
    return radius_km * c



def select_optimum_k(
    scaled_features: np.ndarray,
    scaler: StandardScaler,
    k_values,
    neighbor_count: int = 8,
    movement_band_km=(1.0, 1.5),
    metric_sample_size: int = 6000,
    silhouette_sample_size: int = 3000,
):
    rows = []
    band_low, band_high = movement_band_km

    n_rows = len(scaled_features)
    eval_idx = None

    # Use a fixed subset for metric computation to keep runtime stable on Kaggle.
    if n_rows > metric_sample_size:
        rng = np.random.default_rng(RANDOM_STATE)
        eval_idx = rng.choice(n_rows, size=metric_sample_size, replace=False)

    for k in k_values:
        model = MiniBatchKMeans(
            n_clusters=k,
            n_init=10,
            random_state=RANDOM_STATE,
            batch_size=4096,
            reassignment_ratio=0.01,
        )
        labels = model.fit_predict(scaled_features)

        if eval_idx is None:
            X_eval = scaled_features
            y_eval = labels
        else:
            X_eval = scaled_features[eval_idx]
            y_eval = labels[eval_idx]

        # Clustering-quality metrics on sampled feature space.
        row = {
            "k": k,
            "inertia": model.inertia_,
            "silhouette": np.nan,
            "calinski_harabasz": np.nan,
            "davies_bouldin": np.nan,
            "metrics_eval_rows": int(len(X_eval)),
        }

        unique_clusters = np.unique(y_eval)
        if len(unique_clusters) > 1 and len(X_eval) > 2:
            safe_sil_size = min(silhouette_sample_size, len(X_eval))
            row["silhouette"] = silhouette_score(
                X_eval,
                y_eval,
                sample_size=safe_sil_size,
                random_state=RANDOM_STATE,
            )
            row["calinski_harabasz"] = calinski_harabasz_score(X_eval, y_eval)
            row["davies_bouldin"] = davies_bouldin_score(X_eval, y_eval)

        # Operational metric: driver movement distance in KM between regions.
        centroids_lat_lon = scaler.inverse_transform(model.cluster_centers_)
        distance_matrix = pairwise_haversine_matrix(centroids_lat_lon)
        sorted_distances = np.sort(distance_matrix, axis=1)

        selected_distances = sorted_distances[:, 1 : neighbor_count + 1]
        avg_neighbor_distance_km = selected_distances.mean(axis=1)
        movement_fit_mask = (
            (avg_neighbor_distance_km >= band_low) & (avg_neighbor_distance_km <= band_high)
        )

        row["neighbor_count"] = neighbor_count
        row["movement_band_low_km"] = band_low
        row["movement_band_high_km"] = band_high
        row["avg_neighbor_distance_km"] = float(avg_neighbor_distance_km.mean())
        row["movement_fit_count"] = int(movement_fit_mask.sum())
        row["movement_fit_pct"] = float(movement_fit_mask.mean())

        rows.append(row)

    return pd.DataFrame(rows)



def choose_best_k(
    metrics_df: pd.DataFrame,
    movement_fit_threshold: float = 0.60,
    return_details: bool = False,
):
    ranked = metrics_df.dropna(subset=["silhouette", "calinski_harabasz", "davies_bouldin"]).copy()

    if ranked.empty:
        fallback_row = metrics_df.sort_values("movement_fit_pct", ascending=False).iloc[0]
        best_k = int(fallback_row["k"])
        if return_details:
            return best_k, metrics_df.copy(), True
        return best_k

    feasible = ranked[ranked["movement_fit_pct"] >= movement_fit_threshold].copy()
    used_fallback = False

    if feasible.empty:
        # If strict movement threshold is not met, use top movement-fit candidates.
        feasible = ranked.sort_values(["movement_fit_pct", "silhouette"], ascending=[False, False]).head(3)
        used_fallback = True

    feasible = feasible.copy()
    feasible["rank_movement"] = feasible["movement_fit_pct"].rank(ascending=False, method="dense")
    feasible["rank_silhouette"] = feasible["silhouette"].rank(ascending=False, method="dense")
    feasible["rank_calinski"] = feasible["calinski_harabasz"].rank(ascending=False, method="dense")
    feasible["rank_davies"] = feasible["davies_bouldin"].rank(ascending=True, method="dense")

    feasible["rank_total"] = (
        feasible["rank_movement"]
        + feasible["rank_silhouette"]
        + feasible["rank_calinski"]
        + feasible["rank_davies"]
    )

    feasible_sorted = feasible.sort_values(
        ["rank_total", "movement_fit_pct", "silhouette"],
        ascending=[True, False, False],
    )
    best_k = int(feasible_sorted.iloc[0]["k"])

    if return_details:
        return best_k, feasible_sorted, used_fallback
    return best_k


In [ ]:
scaler_for_search = StandardScaler()
scaled_sample = scaler_for_search.fit_transform(sample_coords)

k_values = list(range(12, 61, 6))
movement_band_km = (1.6, 2.4)
neighbor_count = 8

# Kaggle-safe metric settings (reduce silhouette compute cost)
metric_sample_size = 6000
silhouette_sample_size = 3000

k_metrics = select_optimum_k(
    scaled_features=scaled_sample,
    scaler=scaler_for_search,
    k_values=k_values,
    neighbor_count=neighbor_count,
    movement_band_km=movement_band_km,
    metric_sample_size=metric_sample_size,
    silhouette_sample_size=silhouette_sample_size,
)
k_metrics


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))

sns.lineplot(data=k_metrics, x="k", y="inertia", marker="o", ax=axes[0, 0])
axes[0, 0].set_title("Elbow View (Inertia)")

sns.lineplot(data=k_metrics, x="k", y="silhouette", marker="o", ax=axes[0, 1], color="#16a34a")
axes[0, 1].set_title("Silhouette Score")

sns.lineplot(data=k_metrics, x="k", y="calinski_harabasz", marker="o", ax=axes[0, 2], color="#f97316")
axes[0, 2].set_title("Calinski-Harabasz")

sns.lineplot(data=k_metrics, x="k", y="davies_bouldin", marker="o", ax=axes[1, 0], color="#ef4444")
axes[1, 0].set_title("Davies-Bouldin (Lower is Better)")

sns.lineplot(data=k_metrics, x="k", y="movement_fit_pct", marker="o", ax=axes[1, 1], color="#0ea5e9")
axes[1, 1].set_title(
    f"Movement Fit % (Avg {neighbor_count}-neighbor distance in {movement_band_km[0]}-{movement_band_km[1]} km)"
)
axes[1, 1].set_ylim(0, 1)

sns.lineplot(data=k_metrics, x="k", y="avg_neighbor_distance_km", marker="o", ax=axes[1, 2], color="#7c3aed")
axes[1, 2].axhline(movement_band_km[0], color="gray", linestyle="--", linewidth=1)
axes[1, 2].axhline(movement_band_km[1], color="gray", linestyle="--", linewidth=1)
axes[1, 2].set_title("Mean Neighbor Distance (km)")

plt.tight_layout()
plt.show()


In [ ]:
movement_fit_threshold = 0.60
best_k_auto, feasible_ranked, used_fallback = choose_best_k(
    k_metrics,
    movement_fit_threshold=movement_fit_threshold,
    return_details=True,
)

# Final project decision: use K=30 as the operational sweet spot.
manual_k = 30
available_k = set(k_metrics["k"].tolist())

if manual_k in available_k:
    best_k = manual_k
    selection_mode = "manual_override"
else:
    best_k = best_k_auto
    selection_mode = "auto_fallback"

print(f"Movement band (km): {movement_band_km[0]} to {movement_band_km[1]}")
print(f"Movement fit threshold: {movement_fit_threshold:.0%}")
print(f"Auto-selected k: {best_k_auto}")
print(f"Final selected k: {best_k}")
print(f"Selection mode: {selection_mode}")
print(f"Fallback used by auto-selection: {used_fallback}")

k_metrics.sort_values("k")


## Train Final Scaler + MiniBatchKMeans on Full Data

We use chunked training for scalability.


In [ ]:
# Pass 1: fit scaler on all valid coordinates
scaler = StandardScaler()
valid_rows_for_training = 0

for chunk in coordinate_chunk_reader(input_path, chunksize=250_000):
    cleaned, _ = clean_coordinates(chunk)
    if cleaned.empty:
        continue
    scaler.partial_fit(cleaned)
    valid_rows_for_training += len(cleaned)

print(f"Rows used to fit scaler: {valid_rows_for_training:,}")

# Pass 2: fit MiniBatchKMeans incrementally
cluster_model = MiniBatchKMeans(
    n_clusters=best_k,
    n_init=10,
    random_state=RANDOM_STATE,
    batch_size=8192,
    reassignment_ratio=0.01,
    max_no_improvement=20,
)

for chunk in coordinate_chunk_reader(input_path, chunksize=250_000):
    cleaned, _ = clean_coordinates(chunk)
    if cleaned.empty:
        continue

    scaled_chunk = scaler.transform(cleaned)
    cluster_model.partial_fit(scaled_chunk)

print("Final clustering model trained.")


In [ ]:
# Cluster centers (inverse transformed to original lat/lon)
centroids_lat_lon = scaler.inverse_transform(cluster_model.cluster_centers_)

# Cluster sizes from full data
cluster_sizes = np.zeros(best_k, dtype=np.int64)

for chunk in coordinate_chunk_reader(input_path, chunksize=250_000):
    cleaned, _ = clean_coordinates(chunk)
    if cleaned.empty:
        continue

    labels = cluster_model.predict(scaler.transform(cleaned))
    cluster_sizes += np.bincount(labels, minlength=best_k)

cluster_summary = pd.DataFrame(
    {
        "region_id": np.arange(best_k, dtype=int),
        "center_latitude": centroids_lat_lon[:, 0],
        "center_longitude": centroids_lat_lon[:, 1],
        "cluster_size": cluster_sizes,
    }
)
cluster_summary["cluster_share"] = cluster_summary["cluster_size"] / cluster_summary["cluster_size"].sum()
cluster_summary = cluster_summary.sort_values("cluster_size", ascending=False).reset_index(drop=True)

cluster_summary.head(10)


In [ ]:
# Nearby-region table (supports future relocation recommendations)
distance_matrix = pairwise_haversine_matrix(
    cluster_summary[["center_latitude", "center_longitude"]].to_numpy()
)

nearby_count = 5
neighbor_rows = []

for region_idx in range(distance_matrix.shape[0]):
    distance_row = distance_matrix[region_idx].copy()
    distance_row[region_idx] = np.inf
    nearest_region_ids = np.argsort(distance_row)[:nearby_count]

    for rank, neighbor_id in enumerate(nearest_region_ids, start=1):
        neighbor_rows.append(
            {
                "region_id": int(cluster_summary.iloc[region_idx]["region_id"]),
                "neighbor_region_id": int(cluster_summary.iloc[neighbor_id]["region_id"]),
                "distance_km": float(distance_row[neighbor_id]),
                "neighbor_rank": rank,
            }
        )

region_neighbors = pd.DataFrame(neighbor_rows)
region_neighbors.head(10)


In [ ]:
# Label each trip row with region metadata and save as chunked CSV
output_labeled_path = data_interim / "region_labeled_data.csv"
if output_labeled_path.exists():
    output_labeled_path.unlink()

selected_cols = [
    "tpep_pickup_datetime",
    "pickup_latitude",
    "pickup_longitude",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "passenger_count",
]

header = True
total_labeled_rows = 0

for chunk in full_chunk_reader(input_path, columns=selected_cols, chunksize=200_000):
    if not {"pickup_latitude", "pickup_longitude"}.issubset(chunk.columns):
        continue

    cleaned_coords, valid_mask = clean_coordinates(chunk)
    if cleaned_coords.empty:
        continue

    labeled_chunk = chunk.loc[valid_mask].copy().reset_index(drop=True)
    region_ids = cluster_model.predict(scaler.transform(cleaned_coords))

    labeled_chunk["region_id"] = region_ids.astype("int16")
    labeled_chunk["region_label"] = "R" + labeled_chunk["region_id"].astype(str).str.zfill(2)

    if "tpep_pickup_datetime" in labeled_chunk.columns:
        pickup_dt = pd.to_datetime(labeled_chunk["tpep_pickup_datetime"], errors="coerce")
        labeled_chunk["pickup_hour"] = pickup_dt.dt.hour
        labeled_chunk["pickup_day_of_week"] = pickup_dt.dt.dayofweek
        labeled_chunk["is_weekend"] = pickup_dt.dt.dayofweek >= 5

    if {"total_amount", "trip_distance"}.issubset(labeled_chunk.columns):
        labeled_chunk["fare_efficiency"] = (
            labeled_chunk["total_amount"] / labeled_chunk["trip_distance"].clip(lower=1e-3)
        )

    labeled_chunk.to_csv(
        output_labeled_path,
        mode="w" if header else "a",
        header=header,
        index=False,
    )
    header = False
    total_labeled_rows += len(labeled_chunk)

print(f"Rows written with region labels: {total_labeled_rows:,}")
print(f"Saved: {output_labeled_path}")


In [ ]:
# Build region-hour demand profile + risk baseline for downstream notebooks
pickup_counts_parts = []
fare_eff_parts = []

for chunk in pd.read_csv(
    output_labeled_path,
    usecols=lambda c: c in ["region_id", "pickup_hour", "fare_efficiency"],
    chunksize=250_000,
):
    if {"region_id", "pickup_hour"}.issubset(chunk.columns):
        hourly = (
            chunk.dropna(subset=["region_id", "pickup_hour"])
            .assign(
                region_id=lambda df_: df_["region_id"].astype(int),
                pickup_hour=lambda df_: df_["pickup_hour"].astype(int),
            )
            .groupby(["region_id", "pickup_hour"], as_index=False)
            .size()
            .rename(columns={"size": "pickup_count"})
        )
        pickup_counts_parts.append(hourly)

    if {"region_id", "fare_efficiency"}.issubset(chunk.columns):
        fare_summary = (
            chunk.dropna(subset=["region_id", "fare_efficiency"])
            .assign(region_id=lambda df_: df_["region_id"].astype(int))
            .groupby("region_id", as_index=False)["fare_efficiency"]
            .agg(["sum", "count"])
            .reset_index()
            .rename(columns={"sum": "fare_eff_sum", "count": "fare_eff_count"})
        )
        fare_eff_parts.append(fare_summary)

region_hour_profile = pd.concat(pickup_counts_parts, ignore_index=True)
region_hour_profile = (
    region_hour_profile.groupby(["region_id", "pickup_hour"], as_index=False)["pickup_count"].sum()
)

region_risk = (
    region_hour_profile.groupby("region_id", as_index=False)["pickup_count"]
    .agg(avg_hourly_pickups="mean", std_hourly_pickups="std")
    .fillna({"std_hourly_pickups": 0})
)
region_risk["risk_score"] = (
    region_risk["std_hourly_pickups"] / (region_risk["avg_hourly_pickups"] + 1e-6)
)
region_risk["risk_band"] = pd.cut(
    region_risk["risk_score"],
    bins=[-np.inf, 0.35, 0.75, np.inf],
    labels=["Stable", "Moderate", "Volatile"],
)

region_hour_profile = region_hour_profile.merge(
    region_risk[["region_id", "avg_hourly_pickups", "std_hourly_pickups"]],
    on="region_id",
    how="left",
)
region_hour_profile["surge_threshold"] = (
    region_hour_profile["avg_hourly_pickups"] + 1.5 * region_hour_profile["std_hourly_pickups"]
)
region_hour_profile["historical_surge_flag"] = (
    region_hour_profile["pickup_count"] > region_hour_profile["surge_threshold"]
)

if fare_eff_parts:
    fare_eff_df = pd.concat(fare_eff_parts, ignore_index=True)
    fare_eff_df = (
        fare_eff_df.groupby("region_id", as_index=False)[["fare_eff_sum", "fare_eff_count"]].sum()
    )
    fare_eff_df["avg_fare_efficiency"] = (
        fare_eff_df["fare_eff_sum"] / fare_eff_df["fare_eff_count"].clip(lower=1)
    )

    region_risk = region_risk.merge(
        fare_eff_df[["region_id", "avg_fare_efficiency"]],
        on="region_id",
        how="left",
    )


In [ ]:
# Save all notebook outputs
output_cluster_summary = data_interim / "region_cluster_summary.csv"
output_centroids = data_interim / "region_centroids.csv"
output_neighbors = data_interim / "region_neighbors.csv"
output_hour_profile = data_interim / "region_hour_profile.csv"
output_business_baseline = data_interim / "region_business_baseline.csv"
output_model = models_dir / "region_cluster_pipeline.joblib"

cluster_summary.to_csv(output_cluster_summary, index=False)
cluster_summary[["region_id", "center_latitude", "center_longitude"]].to_csv(
    output_centroids,
    index=False,
)
region_neighbors.to_csv(output_neighbors, index=False)
region_hour_profile.to_csv(output_hour_profile, index=False)
region_risk.to_csv(output_business_baseline, index=False)

joblib.dump(
    {
        "scaler": scaler,
        "cluster_model": cluster_model,
        "k_metrics": k_metrics,
        "selected_k": best_k,
    },
    output_model,
)

print("Saved outputs:")
for path in [
    output_labeled_path,
    output_cluster_summary,
    output_centroids,
    output_neighbors,
    output_hour_profile,
    output_business_baseline,
    output_model,
]:
    print("-", path.relative_to(project_root))


In [ ]:
# Visual validation of region assignments
sample_region_ids = cluster_model.predict(scaler.transform(sample_coords))
plot_df = sample_coords.copy()
plot_df["region_id"] = sample_region_ids

fig, ax = plt.subplots(figsize=(10, 7))
ax.set_facecolor("#0f172a")

ax.scatter(
    plot_df["pickup_longitude"],
    plot_df["pickup_latitude"],
    c=plot_df["region_id"],
    cmap="tab20",
    s=2,
    alpha=0.35,
)

ax.scatter(
    cluster_summary["center_longitude"],
    cluster_summary["center_latitude"],
    marker="x",
    c="white",
    s=120,
    linewidths=2,
    label="Centroids",
)

ax.set_title("NYC Regions from MiniBatchKMeans")
ax.set_xlabel("pickup_longitude")
ax.set_ylabel("pickup_latitude")
ax.legend(loc="upper right")
plt.show()


## Notes
- `region_id` / `region_label` is now available for downstream time-series modeling.
- `region_neighbors.csv` can be used later for relocation scoring (`expected_demand_gain / travel_distance`).
- `region_business_baseline.csv` includes `risk_score` and `risk_band` to support stability insights.
- `region_hour_profile.csv` includes a surge baseline proxy (`mean + 1.5 * std`) for surge flag design.
- `k` is selected using a hybrid rule: movement-distance fit first, then clustering metrics.
- Movement target is set to old-equivalent range: `1.6-2.4 km` for 8-neighbor average distance.
- Final K is fixed to `30` for balance between cluster quality and operational region spacing.
- TODO: Revisit auto-K after end-to-end model results; keep `K=30` for current baseline comparability.
